# Design arm · D2 — Blinded dual coding and adjudication

**Part A** (run once) builds the review set and writes blinded sheets for two reviewers:
- every in-scope event that FDA's root cause leaves as other_unclear
- a stratified random validation sample from each auto-mapped category

Reviewer sheets omit FDA's root cause, the event number, and the reason an event was selected; rows are shuffled separately for each reviewer. coding/review_key.csv links review IDs back to events — **do not share it with reviewers or the adjudicator.**

**Workflow**
1. Run Part A. Send each reviewer their coding/reviewer_X_TO_CODE.csv plus CODEBOOK.md.
2. Reviewers fill category and scope (and notes if needed), then save as coding/reviewer_A_CODED.csv and coding/reviewer_B_CODED.csv.
3. Run Part B: it reports agreement and Cohen's kappa and writes coding/adjudication_TO_CODE.csv.
4. The adjudicator fills final_category (and final_scope), saving as coding/adjudication_DONE.csv.
5. Run Part B again: it writes data/event_cause_final.csv and the FDA-mapping concordance.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from cause_map import CATEGORIES, UNCLEAR, SCOPES

SEED = 20261015
VALIDATION_PER_CATEGORY = 40   # per auto-mapped category (all events if fewer)
CODING = Path("coding"); CODING.mkdir(exist_ok=True)
events = pd.read_csv("data/recall_events.csv", dtype=str)
events["in_scope"] = events["in_scope"].eq("True")
scope = events[events["in_scope"]]
BLIND = ["review_id", "recalling_firm", "product_description", "reason_for_recall", "code_info", "action"]

## Part A — review set and blinded sheets

In [2]:
KEY = CODING / "review_key.csv"
if KEY.exists():
    key = pd.read_csv(KEY, dtype=str)
    print(f"Review set already exists ({len(key)} events) and was not regenerated. Delete design_arm/coding/ to rebuild.")
else:
    rng = np.random.default_rng(SEED)
    parts = [scope[scope["fda_category"] == UNCLEAR].assign(selected_as="all_unclear")]
    for cat in [c for c in CATEGORIES if c != UNCLEAR]:
        pool = scope[scope["fda_category"] == cat]
        n = min(VALIDATION_PER_CATEGORY, len(pool))
        parts.append(pool.sample(n=n, random_state=int(rng.integers(1_000_000_000)))
                         .assign(selected_as="validation_sample"))
    review = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    review["review_id"] = [f"R{i + 1:04d}" for i in range(len(review))]
    key = review[["review_id", "res_event_number", "fda_category", "selected_as"]]
    key.to_csv(KEY, index=False)
    for i, who in enumerate(["A", "B"]):
        sheet = review[BLIND].sample(frac=1, random_state=SEED + 1 + i).copy()
        sheet["category"] = ""
        sheet["scope"] = ""
        sheet["notes"] = ""
        sheet.to_csv(CODING / f"reviewer_{who}_TO_CODE.csv", index=False)
    print(f"Wrote coding/reviewer_A_TO_CODE.csv and reviewer_B_TO_CODE.csv")
print(f"CHECKPOINT  review set {len(key)} events")
print(key.groupby("selected_as")["fda_category"].value_counts().to_string())

Wrote coding/reviewer_A_TO_CODE.csv and reviewer_B_TO_CODE.csv
CHECKPOINT  review set 497 events
selected_as        fda_category         
all_unclear        other_unclear            337
validation_sample  design                    40
                   labeling                  40
                   manufacturing_process     40
                   packaging_sterility       40


## Part B — agreement, adjudication, final codes

In [3]:
def kappa(a, b, labels):
    a, b = list(a), list(b)
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(l) / n) * (b.count(l) / n) for l in labels)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

def load_coded(who):
    p = CODING / f"reviewer_{who}_CODED.csv"
    if not p.exists():
        return None
    d = pd.read_csv(p, dtype=str).fillna("")
    for col in ["category", "scope"]:
        d[col] = d[col].str.strip().str.lower()
    bad = d[~d["category"].isin(CATEGORIES)]
    assert bad.empty, f"reviewer {who}: {len(bad)} rows with missing or invalid category, e.g. {bad['review_id'].head().tolist()}"
    assert set(d["review_id"]) == set(key["review_id"]), f"reviewer {who}: review_ids do not match the review set"
    return d.set_index("review_id")

A, B = load_coded("A"), load_coded("B")
if A is None or B is None:
    print("Waiting for coding/reviewer_A_CODED.csv and coding/reviewer_B_CODED.csv. Part B not run.")
else:
    pair = (key.set_index("review_id")
               .join(A[["category", "scope"]].add_suffix("_A"))
               .join(B[["category", "scope"]].add_suffix("_B")))
    agree = pair["category_A"] == pair["category_B"]
    print(f"CHECKPOINT  category agreement {agree.mean():.1%} | Cohen's kappa {kappa(pair['category_A'], pair['category_B'], CATEGORIES):.3f} | disagreements {int((~agree).sum())}")
    print(f"CHECKPOINT  scope agreement {(pair['scope_A'] == pair['scope_B']).mean():.1%} | kappa {kappa(pair['scope_A'], pair['scope_B'], SCOPES):.3f}")

Waiting for coding/reviewer_A_CODED.csv and coding/reviewer_B_CODED.csv. Part B not run.


In [4]:
ADJ_TODO, ADJ_DONE = CODING / "adjudication_TO_CODE.csv", CODING / "adjudication_DONE.csv"
if A is not None and B is not None:
    dis = pair[~agree].reset_index()
    if not ADJ_TODO.exists():
        ev = events.set_index("res_event_number")
        todo = dis[["review_id", "res_event_number", "category_A", "category_B", "scope_A", "scope_B"]].copy()
        for col in BLIND[1:]:
            todo[col] = todo["res_event_number"].map(ev[col])
        todo = todo.drop(columns="res_event_number")
        todo["final_category"] = ""
        todo["final_scope"] = ""
        todo["adjudication_note"] = ""
        todo.to_csv(ADJ_TODO, index=False)
        print(f"Wrote {ADJ_TODO} ({len(todo)} rows) for the adjudicator.")
    if not ADJ_DONE.exists():
        print("Waiting for coding/adjudication_DONE.csv. Final codes not written.")
    else:
        adj = pd.read_csv(ADJ_DONE, dtype=str).fillna("").set_index("review_id")
        adj["final_category"] = adj["final_category"].str.strip().str.lower()
        adj["final_scope"] = adj["final_scope"].str.strip().str.lower()
        assert adj["final_category"].isin(CATEGORIES).all(), "adjudication: missing or invalid final_category"
        assert set(adj.index) == set(dis["review_id"]), "adjudication rows do not match the disagreements"

        pair["coded_category"] = pair["category_A"].where(agree)
        pair.loc[adj.index, "coded_category"] = adj["final_category"]
        pair["coded_scope"] = pair["scope_A"].where(pair["scope_A"] == pair["scope_B"], "")
        s = adj["final_scope"][adj["final_scope"].ne("")]
        pair.loc[s.index, "coded_scope"] = s

        coded = pair.reset_index().set_index("res_event_number")
        final = events[["res_event_number", "fda_category", "in_scope"]].copy()
        final["manually_coded"] = final["res_event_number"].isin(coded.index)
        final["coded_category"] = final["res_event_number"].map(coded["coded_category"])
        final["coded_scope"] = final["res_event_number"].map(coded["coded_scope"])
        final["final_category"] = final["coded_category"].fillna(final["fda_category"])
        final.to_csv("data/event_cause_final.csv", index=False)
        print(f"CHECKPOINT  wrote data/event_cause_final.csv | manually coded {int(final['manually_coded'].sum())}")

        val = pair[pair["selected_as"] == "validation_sample"]
        print(f"CHECKPOINT  FDA-mapping concordance (validation sample) {(val['coded_category'] == val['fda_category']).mean():.1%} | kappa {kappa(val['fda_category'], val['coded_category'], CATEGORIES):.3f}")
        print(val.groupby("fda_category").apply(lambda g: (g["coded_category"] == g.name).mean()).rename("concordance").to_string())
        print("other_unclear events re-coded as:", pair[pair["selected_as"] == "all_unclear"]["coded_category"].value_counts().to_dict())